# qpl Numerical Toolkit

A compact walkthrough of quadrature, iterative linear solves, and Laplace inversion.

## Setup

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

from qpl.numerics.linear_systems import gauss_seidel_solve, jacobi_solve, sor_solve
from qpl.numerics.quadrature import composite_simpson, composite_trapezoid, gauss_legendre
from qpl.transforms.laplace import inverse_laplace_grid_stehfest
from qpl.utils import choose_by_mode, is_smoke_mode, set_global_seed

SEED = 123
_ = set_global_seed(SEED)
SMOKE_MODE = is_smoke_mode()
plt.style.use("seaborn-v0_8-whitegrid")

N_MATRIX = choose_by_mode(SMOKE_MODE, smoke=40, full=120)
MAX_ITER = choose_by_mode(SMOKE_MODE, smoke=400, full=2_000)
T_GRID_SIZE = choose_by_mode(SMOKE_MODE, smoke=16, full=40)

print(f"seed={SEED} smoke_mode={SMOKE_MODE} n_matrix={N_MATRIX}")

## Quadrature: `exp(x)` on [0, 1]

In [ ]:
def f(x: np.ndarray) -> np.ndarray:
    return np.exp(x)

exact = math.e - 1.0
trap = composite_trapezoid(f, 0.0, 1.0, n_intervals=128)
simp = composite_simpson(f, 0.0, 1.0, n_intervals=128)
gauss = gauss_legendre(f, 0.0, 1.0, n_nodes=16)

print(f"exact={exact:.10f}")
print(f"trap={trap:.10f} abs_err={abs(trap - exact):.3e}")
print(f"simpson={simp:.10f} abs_err={abs(simp - exact):.3e}")
print(f"gauss={gauss:.10f} abs_err={abs(gauss - exact):.3e}")

## Linear Systems: Jacobi vs Gauss-Seidel vs SOR

In [ ]:
A = np.zeros((N_MATRIX, N_MATRIX), dtype=float)
np.fill_diagonal(A, 2.0)
idx = np.arange(N_MATRIX - 1)
A[idx, idx + 1] = -1.0
A[idx + 1, idx] = -1.0
b = np.ones(N_MATRIX, dtype=float)

jac = jacobi_solve(A, b, tol=1e-8, max_iter=MAX_ITER)
gs = gauss_seidel_solve(A, b, tol=1e-8, max_iter=MAX_ITER)
sor = sor_solve(A, b, omega=1.1, tol=1e-8, max_iter=MAX_ITER)

print(f"jacobi_iters={jac.iterations} converged={jac.converged}")
print(f"gs_iters={gs.iterations} converged={gs.converged}")
print(f"sor_iters={sor.iterations} converged={sor.converged}")

plt.figure(figsize=(6.5, 3.5))
plt.semilogy(jac.residual_history, label="Jacobi")
plt.semilogy(gs.residual_history, label="Gauss-Seidel")
plt.semilogy(sor.residual_history, label="SOR (omega=1.1)")
plt.xlabel("iteration")
plt.ylabel("residual norm")
plt.title("Iterative solver residual histories")
plt.legend()
plt.show()

## Laplace Inversion: Recover `exp(-a t)`

In [ ]:
a = 0.75
t_grid = np.linspace(0.2, 3.0, T_GRID_SIZE)

def transform(s: float) -> float:
    return 1.0 / (s + a)

approx = inverse_laplace_grid_stehfest(transform, t_grid, n_terms=10)
truth = np.exp(-a * t_grid)

print(f"max_abs_error={np.max(np.abs(approx - truth)):.6e}")

plt.figure(figsize=(6.5, 3.5))
plt.plot(t_grid, truth, label="truth")
plt.plot(t_grid, approx, linestyle="--", label="Stehfest")
plt.title("Laplace inversion sanity check")
plt.xlabel("t")
plt.ylabel("value")
plt.legend()
plt.show()

## Takeaways

- The numerics module now supports educational solver and quadrature workflows directly.
- Residual history gives a clear convergence diagnostic beyond final residual values.
- Stehfest inversion offers a compact transform-based sanity check.